# Numerical Implementation (Local Jupyter)

Run this notebook in desktop Jupyter with the project environment. It retains local JPG, PDF, and HTML exports.

Set bounds as `[[xmin, xmax], [ymin, ymax]]`, transmissivity as a scalar, and gradient as `(magnitude, azimuth_degrees)`. List wells as `[pumping_rate, x, y]` rows; add as many rows as needed and keep units consistent.

In [ ]:
model_boundaries = [[0, 1000],
                    [0, 500]]
aquifer_transmissivity = .02
regional_gradient = [0.00, 180]
wells = [[1, 100, 50],
         [1, 800, 150],
         [0.1, 250, 420]]

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objs as go
import plotly.offline as py
from IPython.display import HTML, display
from math import cos, pi, radians, sin
from matplotlib.backends.backend_pdf import PdfPages

print("Well Superposition - Zone of Capture")
print("By N. Sitar, UC Berkeley, 09/2022\n")
print("Use consistent units: time in seconds or days, and length in m or ft.\n")

Xmin, Xmax = model_boundaries[0]
Ymin, Ymax = model_boundaries[1]
T = aquifer_transmissivity
gradient, azimuth_degrees = regional_gradient
azimuth = radians(azimuth_degrees)

x = np.arange(Xmin, Xmax, (Xmax - Xmin) / 75.0)
y = np.arange(Ymin, Ymax, (Ymax - Ymin) / 75.0)
X, Y = np.meshgrid(x, y)

S = np.zeros_like(X, dtype=float)
ux = np.zeros_like(X, dtype=float)
uy = np.zeros_like(Y, dtype=float)

for pumping_rate, well_x, well_y in wells:
    dx = X - well_x
    dy = Y - well_y
    radius_squared = dx**2 + dy**2
    S += (pumping_rate / (2 * pi * T)) * np.log(np.sqrt(radius_squared))
    ux -= (pumping_rate / (2 * pi * T)) * dx / radius_squared
    uy -= (pumping_rate / (2 * pi * T)) * dy / radius_squared

S -= gradient * (X * cos(azimuth) + Y * sin(azimuth))
ux += gradient * cos(azimuth)
uy += gradient * sin(azimuth)

matplotlib.rcParams["xtick.direction"] = "in"
matplotlib.rcParams["ytick.direction"] = "in"

width = 10.0
height = (Ymax - Ymin) / (Xmax - Xmin) * width
figure, axes = plt.subplots(figsize=(width, height))
axes.set_xlabel("x", fontsize=16)
axes.set_ylabel("y", fontsize=16)
axes.set_xlim(Xmin, Xmax)
axes.set_ylim(Ymin, Ymax)
axes.streamplot(x, y, ux, uy, density=1, linewidth=1, arrowsize=2, arrowstyle="->")
axes.scatter(
    [well[1] for well in wells],
    [well[2] for well in wells],
    color="#CD2305",
    s=80,
    marker="o",
)
axes.contour(X, Y, S, 15)
axes.set_title(
    "Zone of Capture, Number of Wells = {}, Gradient = {}, Azimuth = {}".format(
        len(wells), gradient, azimuth_degrees
    )
)
figure.savefig("Zone of Capture.jpg")
with PdfPages("Zone of Capture.pdf") as pdf:
    pdf.savefig(figure)

fig = go.Figure(
    go.Surface(
        x=x,
        y=y,
        z=S,
        contours={
            "z": {
                "show": True,
                "highlight": True,
                "color": "white",
                "highlightcolor": "limegreen",
            }
        },
    )
)
fig.update_layout(
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Drawdown"),
    margin=dict(r=20, b=10, l=10, t=10),
)
fig.update_layout(
    xaxis=go.layout.XAxis(title="X-axis"),
    yaxis=go.layout.YAxis(title="Y-axis"),
)
py.plot(fig, filename="Contour.html", auto_open=False)

display(HTML(fig.to_html(include_plotlyjs="cdn")))
plt.show()
plt.close(figure)